preuba de funciones

In [ ]:
from pathlib import Path
import sys

# Get current file location
current_file = Path(__file__).resolve()

# Go up until you find the project root (where "src" exists)
for parent in current_file.parents:
    if (parent / "src").exists():
        project_root = parent
        break

# Add to PYTHONPATH if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now import works
from src.modules import tools_EEG as TEEG

In [ ]:
ACA PONER TODO LO QUE VA ANTES DEL LABELING

In [ ]:
from datetime import timedelta
import pandas as pd
import numpy as np


def label_eeg_windows(
    df_windows,
    df_recordings,
    preictal_range_min=(-10, -5),
    ictal_range_min=(0, 5),
    include_gap_as_interictal=True,
    keep_only_preictal_seizure=False,
    datetime_as_string=False,
    print_summary=True
):
    """
    Label EEG windows as interictal, preictal, or seizure.

    Label mapping:
        interictal = 0
        preictal   = 1
        seizure    = 2

    Parameters
    ----------
    df_windows : pd.DataFrame
        DataFrame with one row per EEG window.
        Expected columns:
            - file_name
            - window_id
            - start_sample
            - end_sample
            - fs
            - seizure_onsets

    df_recordings : pd.DataFrame
        Recording-level metadata DataFrame.
        Expected columns:
            - file_name
            - start_time

    preictal_range_min : tuple
        Time range before seizure onset for preictal labeling, in minutes.
        Example:
            (-10, -5) means from 10 minutes before onset to 5 minutes before onset.

    ictal_range_min : tuple
        Time range around/after seizure onset for seizure labeling, in minutes.
        Example:
            (0, 5) means from seizure onset to 5 minutes after onset.

    include_gap_as_interictal : bool
        If True, windows that are close to a seizure but do not overlap with
        preictal or ictal intervals are labeled as interictal.

        If False, windows between the preictal interval and ictal interval
        are excluded.

        Example with preictal_range_min=(-10, -5) and ictal_range_min=(0, 5):
            onset - 4 min to onset - 1 min is a "gap".
            If True  -> interictal
            If False -> excluded

    keep_only_preictal_seizure : bool
        If True, return only preictal and seizure windows.
        If False, return all labeled windows, including interictal.

    datetime_as_string : bool
        If True, convert window_start_time and window_end_time to strings.
        If False, keep them as pandas datetime objects.

    print_summary : bool
        If True, print label mapping and class counts.

    Returns
    -------
    df_labeled : pd.DataFrame
        DataFrame with labeled EEG windows.
    """

    # -------------------------------
    # Label mapping
    # -------------------------------
    label_map = {
        "interictal": 0,
        "preictal": 1,
        "seizure": 2
    }

    # -------------------------------
    # Helper: overlap function
    # -------------------------------
    def overlaps(a_start, a_end, b_start, b_end):
        """
        Return True if two time intervals overlap.
        """
        return (a_start < b_end) and (a_end > b_start)

    # -------------------------------
    # Helper: check if window is inside the gap between preictal and ictal
    # -------------------------------
    def is_in_gap(window_start, window_end, preictal_end, ictal_start):
        """
        Return True if a window overlaps the gap between the end of the preictal
        interval and the beginning of the ictal interval.
        """
        return overlaps(window_start, window_end, preictal_end, ictal_start)

    # -------------------------------
    # Copy dataframe to avoid overwriting original
    # -------------------------------
    df_labeled = df_windows.copy()

    # Create new columns
    df_labeled["window_start_time"] = pd.NaT
    df_labeled["window_end_time"] = pd.NaT
    df_labeled["class_label"] = np.nan
    df_labeled["label_name"] = pd.NA

    # Optional column to track excluded gap windows
    df_labeled["excluded_reason"] = pd.NA

    # -------------------------------
    # Main labeling loop
    # -------------------------------
    for idx, row in df_labeled.iterrows():

        # Match recording-level metadata
        matching_rec = df_recordings[df_recordings["file_name"] == row["file_name"]]

        if matching_rec.empty:
            raise ValueError(
                f"No matching recording found in df_recordings for file_name: {row['file_name']}"
            )

        rec = matching_rec.iloc[0]

        # Recording start time
        recording_start = pd.to_datetime(rec["start_time"])
        fs = row["fs"]

        # Convert sample indices to seconds
        start_sec = row["start_sample"] / fs
        end_sec = row["end_sample"] / fs

        # Compute real datetime of each window
        window_start_time = recording_start + pd.Timedelta(seconds=start_sec)
        window_end_time = recording_start + pd.Timedelta(seconds=end_sec)

        # Save window datetime information
        df_labeled.at[idx, "window_start_time"] = window_start_time
        df_labeled.at[idx, "window_end_time"] = window_end_time

        # Default label = interictal
        assigned_label = label_map["interictal"]
        assigned_name = "interictal"
        excluded_reason = pd.NA

        # Clean seizure_onsets
        seizure_onsets = clean_onsets(row["seizure_onsets"])

        # If there are no seizure onsets, keep the default label: interictal
        if len(seizure_onsets) == 0:
            df_labeled.at[idx, "class_label"] = assigned_label
            df_labeled.at[idx, "label_name"] = assigned_name
            continue

        # Flag used only if include_gap_as_interictal=False
        overlaps_gap = False

        # Loop through every seizure onset associated with this window/recording
        for onset in seizure_onsets:

            onset = pd.to_datetime(onset)

            # Define preictal interval
            preictal_start = onset + pd.Timedelta(minutes=preictal_range_min[0])
            preictal_end = onset + pd.Timedelta(minutes=preictal_range_min[1])

            # Define ictal/seizure interval
            seizure_start = onset + pd.Timedelta(minutes=ictal_range_min[0])
            seizure_end = onset + pd.Timedelta(minutes=ictal_range_min[1])

            # -------------------------------
            # 1) Seizure has highest priority
            # -------------------------------
            if overlaps(window_start_time, window_end_time, seizure_start, seizure_end):
                assigned_label = label_map["seizure"]
                assigned_name = "seizure"
                excluded_reason = pd.NA
                break

            # -------------------------------
            # 2) Preictal has second priority
            # -------------------------------
            elif overlaps(window_start_time, window_end_time, preictal_start, preictal_end):
                assigned_label = label_map["preictal"]
                assigned_name = "preictal"
                excluded_reason = pd.NA

            # -------------------------------
            # 3) Optional: detect gap between preictal and seizure
            # -------------------------------
            elif preictal_end < seizure_start:
                if is_in_gap(window_start_time, window_end_time, preictal_end, seizure_start):
                    overlaps_gap = True

        # -------------------------------
        # Exclude gap windows if requested
        # -------------------------------
        if overlaps_gap and not include_gap_as_interictal:
            assigned_label = np.nan
            assigned_name = pd.NA
            excluded_reason = "periictal_gap"

        # Save final assigned label
        df_labeled.at[idx, "class_label"] = assigned_label
        df_labeled.at[idx, "label_name"] = assigned_name
        df_labeled.at[idx, "excluded_reason"] = excluded_reason

    # -------------------------------
    # Remove excluded windows, if any
    # -------------------------------
    df_labeled = df_labeled[df_labeled["class_label"].notna()].copy()

    # -------------------------------
    # Optional: keep only preictal and seizure windows
    # -------------------------------
    if keep_only_preictal_seizure:
        df_labeled = df_labeled[
            df_labeled["label_name"].isin(["preictal", "seizure"])
        ].copy()

    # Make class_label integer
    df_labeled["class_label"] = df_labeled["class_label"].astype(int)

    # -------------------------------
    # Optional: convert datetime columns to string format
    # -------------------------------
    if datetime_as_string:
        df_labeled["window_start_time"] = pd.to_datetime(
            df_labeled["window_start_time"]
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

        df_labeled["window_end_time"] = pd.to_datetime(
            df_labeled["window_end_time"]
        ).dt.strftime("%Y-%m-%d %H:%M:%S")

    # -------------------------------
    # Print summary
    # -------------------------------
    if print_summary:
        print("Label mapping:")
        print("  0 = interictal")
        print("  1 = preictal")
        print("  2 = seizure")

        print("\nClass counts:")
        print(df_labeled["label_name"].value_counts(dropna=False))

        print("\nClass counts by numeric label:")
        print(df_labeled["class_label"].value_counts().sort_index())

    return df_labeled